### This script is for extractionand preparing the dataset (preprocessing). 

**Topic**: Redshift Estiamtion of BL Lacs using ML concepts

**Objects**:  BL Lac objects (mostly AGN sources)


##### Datasets and Catalogues: Fermi-LAT 4FGL-DR4 and 4LAC-DR4 catalogs

### Procedure followed:
1. Downloading and inspecting master FITS file of Fermi-LAT 4FGL-DR4.
2. Filtering for Blazar Classes.
3. Constructing flat dataframe on applied filter.
4. Saving filtered dataset to csv.
5. The physical redshifts ($z$) are stored in the companion 4LAC catalog. We merge this on `Source_Name`. Note the use of `.astype(float)` to bypass byte-order (endianness) compatibility issues between FITS and the pandas backend. Hence, we split the data into training and Target dataset: Objects with a known redshift become our Training Set. The featureless objects missing a redshift become our Target (Inference) Set.

#### Step 1: Doweloading and inspecting master FITS file of Fermi-LAT 4FGL-DR4.

In [1]:
from astropy.io import fits
import pandas as pd
import numpy as np

# 1. Open the binary table extension
hdul = fits.open('../data/raw/gll_psc_v35.fit')
data = hdul[1].data

#### Step 2: Filtering for Blazar Classes

In [2]:
# 2. Filter for Blazar classes
classes = np.char.strip(data['CLASS1'])
mask = np.isin(classes, ['bll', 'BLL', 'fsrq', 'FSRQ', 'bcu', 'BCU'])
blazar_data = data[mask]

#### Step 3: Constructing flat dataframe on this applied filter.

In [3]:
# 3. Construct the flat DataFrame
df = pd.DataFrame({
    'Source_Name': np.char.strip(blazar_data['Source_Name']),
    'RAJ2000': blazar_data['RAJ2000'],
    'DEJ2000': blazar_data['DEJ2000'],
    'CLASS1': classes[mask],
    'Signif_Avg': blazar_data['Signif_Avg'],
    'PL_Index': blazar_data['PL_Index'],
    'LP_Index': blazar_data['LP_Index'],
    'LP_beta': blazar_data['LP_beta'],
    'Energy_Flux100': blazar_data['Energy_Flux100'],
    'Variability_Index': blazar_data['Variability_Index']
})

# Unpack the 8-band flux arrays into separate columns
flux_bands = blazar_data['Flux_Band']
for i in range(8):
    df[f'Flux_Band_{i+1}'] = flux_bands[:, i]

#### 4. Saving filtered dataframe to csv

In [4]:
# 4. Save to CSV for the ML pipeline
df.to_csv('../data/interim/4fgl_dr4_blazar_features.csv', index=False)
hdul.close()

#### Step 5. The physical redshifts ($z$) are stored in the companion 4LAC catalog. We merge this on `Source_Name`. Note the use of `.astype(float)` to bypass byte-order (endianness) compatibility issues between FITS and the pandas backend. Hence, we split the data into training and Target dataset: Objects with a known redshift become our Training Set. The featureless objects missing a redshift become our Target (Inference) Set.

In [9]:
from astropy.io import fits
import pandas as pd
import numpy as np

# 1. Load the features CSV
features_df = pd.read_csv('../data/interim/4fgl_dr4_blazar_features.csv')

# 2. Load the 4LAC-DR4 Catalog (High-Latitude)
hdul_4lac = fits.open('../data/raw/table_4LAC.fits')
data_4lac = hdul_4lac[1].data

In [11]:
# 3. Extract columns and force native byte order using astype()
df_4lac = pd.DataFrame({
    'Source_Name': np.char.strip(data_4lac['Source_Name']),
    # .astype(float) forces a completely new, native little-endian array
    'Redshift': data_4lac['Redshift'].astype(float)
})

# Replace Fermi placeholder values (-1.0 or 0.0) with NaN
df_4lac['Redshift'] = df_4lac['Redshift'].replace({-1.0: np.nan, 0.0: np.nan})

In [12]:
# 4. Merge the two DataFrames on 'Source_Name'
merged_df = pd.merge(features_df, df_4lac, on='Source_Name', how='left')

In [13]:
# 5. Split into Training (known z) and Target (unknown z) sets
train_df = merged_df[merged_df['Redshift'].notna()]
target_df = merged_df[merged_df['Redshift'].isna()]

print(f"Training set size: {len(train_df)}")
print(f"Target set size: {len(target_df)}")

# Save them out
train_df.to_csv('../data/interim/blazar_training_set.csv', index=False)
target_df.to_csv('../data/interim/blazar_target_set.csv', index=False)
hdul_4lac.close()

Training set size: 2777
Target set size: 1156


##### **Conclusion:** Isolating the `bll`, `fsrq`, and `bcu` classes from the 4FGL FITS catalog narrows the dataset specifically to the active galactic nuclei relevant to this study. Unpacking the nested 8-band gamma-ray flux arrays into discrete scalar columns transforms the raw hierarchical data into a flat, tabular format. This structural conversion is a critical prerequisite, rendering the photometry directly compatible with standard tree-based machine learning algorithms and staging the raw data for the extraction of Extragalactic Background Light (EBL) attenuation signatures in the next phase.